# 05 - Wave Equation

In this notebook, we solve the wave equation using PINNs:

$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}$$

This equation models wave propagation, vibrations, acoustics, and electromagnetics.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
from pinns import Config, MLP, Trainer, set_seed
from pinns.equations.wave import WaveEquation1D
from pinns.losses import create_pinn_loss
from pinns.sampling import UniformSampler
from pinns.evaluation import plot_solution_1d, compute_all_metrics

In [ ]:
set_seed(42)

config = Config(
    name='wave_1d',
    equation_name='wave_1d',
    equation_params={'c': 1.0, 'length': 1.0, 't_max': 1.0},
    model={'type': 'mlp', 'hidden_layers': [32, 32, 32], 'activation': 'tanh'},
    training={'n_collocation': 1000, 'n_boundary': 100, 'n_initial': 100, 'epochs': 3000, 'lr': 1e-3},
)

eq = WaveEquation1D(c=1.0)
print(f"Domain: {eq.domain}")

In [ ]:
model = MLP(
    input_dim=2,  # [x, t]
    output_dim=1,
    hidden_layers=[32, 32, 32],
    activation='tanh'
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
trainer = Trainer(
    model=model,
    equation=eq,
    config=config,
    print_every=500
)

history = trainer.train()

plt.figure(figsize=(10, 4))
plt.plot(history['total_loss'], label='Total Loss')
plt.plot(history['physics_loss'], label='Physics Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.yscale('log')
plt.legend()
plt.title('Wave Equation Training')
plt.show()

In [ ]:
import numpy as np

x = torch.linspace(0, 1, 100).reshape(-1, 1)
t = torch.full_like(x, 0.5)

with torch.no_grad():
    u_pred = model(torch.cat([x, t], dim=1))

# Analytical solution for c=1: u(x,t) = sin(pi*x) * cos(pi*t)
u_exact = torch.sin(np.pi * x) * torch.cos(np.pi * t)

plot_solution_1d(
    x=x.numpy(),
    u_pred=u_pred.numpy(),
    u_true=u_exact.numpy(),
    title='Wave Equation at t=0.5'
)

In [ ]:
metrics = compute_all_metrics(u_pred.numpy(), u_exact.numpy())
print("\nError Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.6e}")